In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns





In [ ]:
# Load the dataset

df = pd.read_csv('../data/raw/ecommerce_customer_data_custom_ratios.csv')


In [12]:
# Basic info about the dataset
print("Dataset shape:", df.shape)
print("\nColumn names and types:")
print(df.dtypes)
print("\nFirst few rows:")
df.head()

Dataset shape: (250000, 13)

Column names and types:
Customer ID                int64
Purchase Date             object
Product Category          object
Product Price              int64
Quantity                   int64
Total Purchase Amount      int64
Payment Method            object
Customer Age               int64
Returns                  float64
Customer Name             object
Age                        int64
Gender                    object
Churn                      int64
dtype: object

First few rows:


,Customer ID,Purchase Date,Product Category,Product Price,Quantity,Total Purchase Amount,Payment Method,Customer Age,Returns,Customer Name,Age,Gender,Churn
0,46251,2020-09-08 09:38:32,Electronics,12,3,740,Credit Card,37,0.0,Christine Hernandez,37,Male,0
1,46251,2022-03-05 12:56:35,Home,468,4,2739,PayPal,37,0.0,Christine Hernandez,37,Male,0
2,46251,2022-05-23 18:18:01,Home,288,2,3196,PayPal,37,0.0,Christine Hernandez,37,Male,0
3,46251,2020-11-12 13:13:29,Clothing,196,1,3509,PayPal,37,0.0,Christine Hernandez,37,Male,0
4,13593,2020-11-27 17:55:11,Home,449,1,3452,Credit Card,49,0.0,James Grant,49,Female,1


In [13]:
# Check for missing values
print("Missing values per column:")
missing_values = df.isnull().sum()
print(missing_values)

Missing values per column:
Customer ID                  0
Purchase Date                0
Product Category             0
Product Price                0
Quantity                     0
Total Purchase Amount        0
Payment Method               0
Customer Age                 0
Returns                  47596
Customer Name                0
Age                          0
Gender                       0
Churn                        0
dtype: int64


In [17]:
# Percentage of missing values
missing_percentage = (missing_values / len(df)) * 100
print("\nMissing values percentage:")
print(missing_percentage[missing_percentage > 0])



Missing values percentage:
Returns    19.0384
dtype: float64


In [20]:
#Check for duplicate columns
print("Duplicate columns check:")
print("Customer Age vs Age:", df['Customer Age'].equals(df['Age']))

Duplicate columns check:
Customer Age vs Age: True


In [21]:
# Drop redundant column
df = df.drop('Customer Age', axis=1)
print("Dropped 'Customer Age' column")

Dropped 'Customer Age' column


In [22]:
# Convert Purchase Date to datetime
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'])




In [23]:
# Check if numeric columns are properly typed
numeric_columns = ['Product Price', 'Quantity', 'Total Purchase Amount', 'Age', 'Returns', 'Churn']
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Data types after conversion:")
print(df.dtypes)

Data types after conversion:
Customer ID                       int64
Purchase Date            datetime64[ns]
Product Category                 object
Product Price                     int64
Quantity                          int64
Total Purchase Amount             int64
Payment Method                   object
Returns                         float64
Customer Name                    object
Age                               int64
Gender                           object
Churn                             int64
dtype: object


In [24]:
# Strategy for Returns column (has missing values)
print("Returns column missing values:", df['Returns'].isnull().sum())






Returns column missing values: 47596


In [25]:
# Option 1: Fill with 0 (assuming no return if not specified)
df['Returns'] = df['Returns'].fillna(0)

In [26]:
# For any other missing values, decide based on context
# Check if there are any remaining missing values
print("Remaining missing values:")
print(df.isnull().sum())

Remaining missing values:
Customer ID              0
Purchase Date            0
Product Category         0
Product Price            0
Quantity                 0
Total Purchase Amount    0
Payment Method           0
Returns                  0
Customer Name            0
Age                      0
Gender                   0
Churn                    0
dtype: int64


In [27]:
# Check for data inconsistencies
print("Unique values in categorical columns:")
print("Product Category:", df['Product Category'].unique())
print("Payment Method:", df['Payment Method'].unique())
print("Gender:", df['Gender'].unique())

Unique values in categorical columns:
Product Category: ['Electronics' 'Home' 'Clothing' 'Books']
Payment Method: ['Credit Card' 'PayPal' 'Cash' 'Crypto']
Gender: ['Male' 'Female']


In [28]:
# Check for logical inconsistencies
print("\nLogical checks:")
print("Negative prices:", (df['Product Price'] < 0).sum())
print("Negative quantities:", (df['Quantity'] < 0).sum())
print("Age range:", df['Age'].min(), "to", df['Age'].max())


Logical checks:
Negative prices: 0
Negative quantities: 0
Age range: 18 to 70


In [30]:
# Identify outliers using IQR method
def find_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers

In [31]:
# Check for outliers in key numeric columns
for col in ['Product Price', 'Quantity', 'Total Purchase Amount']:
    outliers = find_outliers(df, col)
    print(f"Outliers in {col}: {len(outliers)}")

Outliers in Product Price: 0
Outliers in Quantity: 0
Outliers in Total Purchase Amount: 0


In [33]:
# Check if Total Purchase Amount = Product Price * Quantity
df['Calculated_Total'] = df['Product Price'] * df['Quantity']
df['Amount_Difference'] = df['Total Purchase Amount'] - df['Calculated_Total']

print("Records where Total Amount doesn't match Price * Quantity:")
inconsistent_amounts = df[abs(df['Amount_Difference']) > 0.01]  # Allow small rounding differences
print(f"Found {len(inconsistent_amounts)} inconsistent records")

if len(inconsistent_amounts) > 0:
    print(inconsistent_amounts[['Product Price', 'Quantity', 'Total Purchase Amount', 'Calculated_Total']].head())


Records where Total Amount doesn't match Price * Quantity:
Found 249952 inconsistent records
   Product Price  Quantity  Total Purchase Amount  Calculated_Total
0             12         3                    740                36
1            468         4                   2739              1872
2            288         2                   3196               576
3            196         1                   3509               196
4            449         1                   3452               449


In [ ]:
# Check for potential gender inconsistencies based on names

name_gender_check = df.groupby('Customer Name')['Gender'].unique()
print("Customers with multiple genders recorded:")
for name, genders in name_gender_check.items():
    if len(genders) > 1:
        print(f"{name}: {genders}")


corrections = {
    'Christine Hernandez': 'Female',  # Christine is typically female
    'James Grant': 'Male'  # James is typically male
}

for name, correct_gender in corrections.items():
    df.loc[df['Customer Name'] == name, 'Gender'] = correct_gender
    print(f"Corrected gender for {name} to {correct_gender}")


Customers with multiple genders recorded:
Aaron Curry: ['Female' 'Male']
Aaron Ford: ['Female' 'Male']
Aaron Kelley: ['Male' 'Female']
Aaron Lee: ['Female' 'Male']
Aaron Moore: ['Male' 'Female']
Aaron Smith: ['Male' 'Female']
Aaron Williams: ['Male' 'Female']
Aaron Wilson: ['Female' 'Male']
Aaron Wolf: ['Male' 'Female']
Abigail Allen: ['Female' 'Male']
Adam Baker: ['Female' 'Male']
Adam Clark: ['Male' 'Female']
Adam Davis: ['Male' 'Female']
Adam Higgins: ['Female' 'Male']
Adam Hill: ['Male' 'Female']
Adam Johnson: ['Male' 'Female']
Adam Jones: ['Male' 'Female']
Adam King: ['Female' 'Male']
Adam Rodriguez: ['Female' 'Male']
Adam Scott: ['Male' 'Female']
Adam Smith: ['Male' 'Female']
Adam Williams: ['Female' 'Male']
Adam Young: ['Male' 'Female']
Aimee Johnson: ['Female' 'Male']
Alan Coleman: ['Female' 'Male']
Alan Moore: ['Female' 'Male']
Albert Johnson: ['Male' 'Female']
Alex Gonzalez: ['Male' 'Female']
Alexander Brock: ['Female' 'Male']
Alexander Brown: ['Female' 'Male']
Alexander Davi

In [ ]:
import os
import pandas as pd

print("=== FINAL DATA QUALITY SUMMARY ===")
print(f"Dataset shape: {df.shape}")


df.columns = df.columns.str.strip()


if "Purchase Date" in df.columns:
    df["Purchase Date"] = pd.to_datetime(df["Purchase Date"], errors="coerce")
    date_range = f"{df['Purchase Date'].min()} to {df['Purchase Date'].max()}"
else:
    date_range = "Purchase Date column not found"

print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Date range: {date_range}")

if "Customer ID" in df.columns:
    print(f"Unique customers: {df['Customer ID'].nunique()}")
else:
    print("Unique customers: Customer ID column not found")

if "Product Category" in df.columns:
    print(f"Unique products categories: {df['Product Category'].nunique()}")
else:
    print("Unique products categories: Product Category column not found")


os.makedirs("data/cleaned", exist_ok=True)

# this Save cleaned data
df.to_csv("data/cleaned/ecommerce_customer_data_cleaned.csv", index=False)
print("\nCleaned data saved to 'data/cleaned/ecommerce_customer_data_cleaned.csv'")

=== FINAL DATA QUALITY SUMMARY ===
Dataset shape: (250000, 14)
Missing values: 0
Duplicate rows: 0
Date range: 2020-01-01 00:15:00 to 2023-09-15 12:24:08
Unique customers: 49673
Unique products categories: 4

Cleaned data saved to 'data/cleaned/ecommerce_customer_data_cleaned.csv'


In [ ]:
#  this create a data quality report
quality_report = {
    'total_records': len(df),
    'missing_values': df.isnull().sum().to_dict(),
    'data_types': df.dtypes.to_dict(),
    'unique_customers': df['Customer ID'].nunique(),
    'date_range': {
        'start': df['Purchase Date'].min(),
        'end': df['Purchase Date'].max()
    },
    'value_ranges': {
        'age': {'min': df['Age'].min(), 'max': df['Age'].max()},
        'price': {'min': df['Product Price'].min(), 'max': df['Product Price'].max()}
    }
}

print("Data Quality Report:")
for key, value in quality_report.items():
    print(f"{key}: {value}")


Data Quality Report:
total_records: 250000
missing_values: {'Customer ID': 0, 'Purchase Date': 0, 'Product Category': 0, 'Product Price': 0, 'Quantity': 0, 'Total Purchase Amount': 0, 'Payment Method': 0, 'Returns': 0, 'Customer Name': 0, 'Age': 0, 'Gender': 0, 'Churn': 0, 'Calculated_Total': 0, 'Amount_Difference': 0}
data_types: {'Customer ID': dtype('int64'), 'Purchase Date': dtype('<M8[ns]'), 'Product Category': dtype('O'), 'Product Price': dtype('int64'), 'Quantity': dtype('int64'), 'Total Purchase Amount': dtype('int64'), 'Payment Method': dtype('O'), 'Returns': dtype('float64'), 'Customer Name': dtype('O'), 'Age': dtype('int64'), 'Gender': dtype('O'), 'Churn': dtype('int64'), 'Calculated_Total': dtype('int64'), 'Amount_Difference': dtype('int64')}
unique_customers: 49673
date_range: {'start': Timestamp('2020-01-01 00:15:00'), 'end': Timestamp('2023-09-15 12:24:08')}
value_ranges: {'age': {'min': 18, 'max': 70}, 'price': {'min': 10, 'max': 500}}
